# Comparison: SAC, V-CARA without counterfactuals, full V-CARA

Reads `results/metrics/eval_*.json` written by `scripts/evaluate.py`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
variants = ["sac", "vcara_no_cf", "vcara"]
rows = []
for v in variants:
    p = ROOT / "results" / "metrics" / f"eval_{v}.json"
    if not p.exists():
        print("missing", p.name); continue
    d = json.loads(p.read_text())
    for k, a in d["aggregate"].items():
        rows.append(dict(variant=v, metric=k, mean=a["mean"], sd=a["sd_across_seeds"], n=a["n"]))
tab = pd.DataFrame(rows)
tab

In [ ]:
keys = ["mean_return", "mean_coverage", "off_road_frac", "completion"]
sub = tab[tab["metric"].isin(keys)]
fig, axes = plt.subplots(1, len(keys), figsize=(14, 3.5))
for ax, k in zip(axes, keys):
    g = sub[sub["metric"] == k]
    ax.bar(g["variant"], g["mean"], yerr=g["sd"].fillna(0), capsize=4)
    ax.set_title(k)
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Held-out evaluation, mean +- across-seed SD")
fig.tight_layout()

## Reading the comparison

Each bar is the across-seed mean; error bars are the across-seed sample SD. Same seeds and same
evaluation seeds for all variants. With few seeds, a difference smaller than the SD should not be
reported as an effect.